In [6]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import random
from itertools import product
import optuna
import numpy as np
from scipy.stats import norm
import statsmodels.api as sm
import statsmodels.formula.api as smf
from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search, estimate_single_config

In [3]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)
stock_data = pd.read_csv("../../data/X.csv", index_col=0, parse_dates=True)

Summary Statistics of Data


In [210]:
# boolean mask: columns that contain any letter
has_letters = stock_data.columns.str.contains('[a-zA-Z]')

# topic-related columns (strings, topic names, etc.)
topic_data = stock_data.loc[:, has_letters].copy()

# stock-related columns (numeric identifiers, PERMNOs, etc.)
stock_data_only = stock_data.loc[:, ~has_letters].copy()

In [211]:
stock_data_only.shape

(1889, 1620)

In [ ]:
def to_ar1_innovations(
    X: pd.DataFrame,
    min_obs: int = 30,
    print_ar1_summary: bool = True,
    return_ar1_coefs: bool = False
):
    """
    Return AR(1) innovations (residuals) for each column of X.

    Also computes the AR(1) coefficient (slope on lagged x) per column and
    prints summary statistics for these coefficients.

    Returns:
      - X_innov if return_ar1_coefs is False
      - (X_innov, ar1_phi) if return_ar1_coefs is True, where ar1_phi is a Series of slopes.
    """
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")
    ar1_phi = {}  # store phi_hat per column

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()

        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()

        # store residuals (innovations)
        X_innov.loc[tmp.index, col] = res.resid

        # store AR(1) slope coefficient
        ar1_phi[col] = float(res.params["x_lag1"])

    ar1_phi = pd.Series(ar1_phi, name="ar1_phi")

    if print_ar1_summary:
        if ar1_phi.empty:
            print("AR(1) coefficient summary: no columns passed the min_obs/variation filters.")
        else:
            summary = pd.Series({
                "N_topics": int(ar1_phi.count()),
                "Mean": float(ar1_phi.mean()),
                "Std. Dev.": float(ar1_phi.std()),
                "Min": float(ar1_phi.min()),
                "P1": float(ar1_phi.quantile(0.01)),
                "P5": float(ar1_phi.quantile(0.05)),
                "Median": float(ar1_phi.median()),
                "P95": float(ar1_phi.quantile(0.95)),
                "P99": float(ar1_phi.quantile(0.99)),
                "Max": float(ar1_phi.max()),
                "Frac >= 0.8": float((ar1_phi >= 0.8).mean()),
                "Frac >= 0.9": float((ar1_phi >= 0.9).mean()),
            })
            print("\nAR(1) coefficient (phi) summary across columns:")
            print(summary.to_string(float_format=lambda x: f"{x:0.4f}"))

    return (X_innov, ar1_phi) if return_ar1_coefs else X_innov


def panel_construction(stock_data: pd.DataFrame) -> pd.DataFrame:
    """
    Produce Panel A summary statistics for log stock returns:
    r_{i,t} = log(1 + R_{i,t})
    """

    # 1) log-transform (safe for CRSP-style returns)
    log_ret = np.log1p(stock_data)

    # 2) stack to full panel
    r = log_ret.stack().rename("log_return")

    panel = pd.DataFrame({
        "Mean":        [r.mean()],
        "Std. Dev.":   [r.std()],
        "Min":         [r.min()],
        "Max":         [r.max()],
        "Skewness":    [r.skew()],
        "Kurtosis":    [r.kurtosis()],  # excess kurtosis
        "Obs.":        [r.count()]
    })

    return panel



def panel_b_ar1_phi(ar1_phi: pd.Series) -> pd.DataFrame:
    """
    Panel B summary stats for AR(1) coefficients (phi) across topics.
    Expects a Series indexed by topic name with values = phi_hat.
    """
    s = pd.to_numeric(ar1_phi, errors="coerce").dropna()

    panel_b = pd.DataFrame({
        "Mean":      [s.mean()],
        "Std. Dev.": [s.std()],
        "Min":       [s.min()],
        "Max":       [s.max()],
        "Skewness":  [s.skew()],
        "Kurtosis":  [s.kurtosis()],   # excess kurtosis
        "Obs.":      [s.count()],
    }, index=[r"AR(1) coefficient $\phi$"])

    return panel_b


In [ ]:
print("Topic data summary:")
panel_a = panel_construction(topic_data)
print(panel_a.round(4))

print("Retun data summary:")
panel_a = panel_construction(stock_data_only)
print(panel_a.round(4))

print("AR1 coefficients summary:")
X_innov, ar1_phi = to_ar1_innovations(topic_data, return_ar1_coefs=True)
panel_b = panel_b_ar1_phi(ar1_phi)

Corrleation Matrices and Coefficients

In [ ]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# INPUT ASSUMPTION
# ------------------------------------------------------------
# You already have your data in a pandas DataFrame called df
# - index: DatetimeIndex named "date"
# - columns: 180 topic series (floats)
#
# Example:
# df = your_dataframe
# df.index = pd.to_datetime(df.index)
# df = df.sort_index()
# ------------------------------------------------------------

def analyze_correlations(df: pd.DataFrame, max_lag: int = 12, print_n: int = 20):
    # ---- basic checks / cleanup
    df = df.copy()
    if "date" in df.columns and not isinstance(df.index, pd.DatetimeIndex):
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")

    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("df must have a DatetimeIndex (or a 'date' column).")

    df = df.sort_index()

    # keep numeric columns only (should be all 180)
    X = df.apply(pd.to_numeric, errors="coerce")
    # drop columns that are completely NaN
    X = X.dropna(axis=1, how="all")

    # ------------------------------------------------------------
    # 1) Contemporaneous correlation matrix across the 180 series
    # ------------------------------------------------------------
    corr_mat = X.corr()  # Pearson by default
    print("\n=== Contemporaneous correlation matrix (across series) ===")
    print(f"Shape: {corr_mat.shape}")
    print(corr_mat.round(4))

    # "Print the correlation coefficient" (summary of off-diagonal correlations)
    off_diag = corr_mat.where(~np.eye(corr_mat.shape[0], dtype=bool)).stack()
    print("\n=== Summary of pairwise (off-diagonal) correlations ===")
    print(f"N pairs: {off_diag.shape[0]:,}")
    print(f"Mean:    {off_diag.mean(): .6f}")
    print(f"Median:  {off_diag.median(): .6f}")
    print(f"Std:     {off_diag.std(): .6f}")
    print(f"Min:     {off_diag.min(): .6f}")
    print(f"Max:     {off_diag.max(): .6f}")

    # Top absolute correlations (most extreme pairs)
    top_abs = off_diag.abs().sort_values(ascending=False).head(print_n)
    print(f"\n=== Top {print_n} absolute pairwise correlations (contemporaneous) ===")
    for (c1, c2), val_abs in top_abs.items():
        val = corr_mat.loc[c1, c2]
        print(f"{c1}  vs  {c2}:  corr = {val: .6f}")

    # ------------------------------------------------------------
    # 2) Correlation between lags of the 180 series
    #    For each lag k: corr(X_t, X_{t-k}) across series, i.e. Corr(X, X.shift(k))
    # ------------------------------------------------------------
    lag_corr = pd.DataFrame(index=X.columns, columns=range(1, max_lag + 1), dtype=float)

    for k in range(1, max_lag + 1):
        X_lag = X.shift(k)
        # for each column, correlate with its own lag
        for col in X.columns:
            s0 = X[col]
            s1 = X_lag[col]
            lag_corr.loc[col, k] = s0.corr(s1)  # automatically pairwise drops NaNs

    print("\n=== Autocorrelation (each series with its own lag) ===")
    print(f"Rows: {lag_corr.shape[0]}, Lags: {lag_corr.shape[1]}")
    print(lag_corr.round(4))

    # Summaries per lag
    print("\n=== Autocorrelation summary by lag ===")
    summary_by_lag = pd.DataFrame(
        {
            "mean": lag_corr.mean(axis=0),
            "median": lag_corr.median(axis=0),
            "std": lag_corr.std(axis=0),
            "min": lag_corr.min(axis=0),
            "max": lag_corr.max(axis=0),
        }
    )
    print(summary_by_lag.round(6))

    # Most persistent / least persistent at lag 1 (useful quick diagnostic)
    if 1 in lag_corr.columns:
        lag1 = lag_corr[1].dropna()
        print(f"\n=== Top {min(print_n, len(lag1))} autocorrelations at lag 1 ===")
        for name, val in lag1.sort_values(ascending=False).head(print_n).items():
            print(f"{name}:  rho(1) = {val: .6f}")

        print(f"\n=== Bottom {min(print_n, len(lag1))} autocorrelations at lag 1 ===")
        for name, val in lag1.sort_values(ascending=True).head(print_n).items():
            print(f"{name}:  rho(1) = {val: .6f}")

    return corr_mat, lag_corr, summary_by_lag


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_corr_heatmap_png(
    df: pd.DataFrame,
    output_file: str = "correlation_heatmap.png",
    figsize=(10, 10),
    cmap="RdBu_r",
    dpi=600,
):
    # keep numeric columns only
    X = df.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")

    # correlation matrix
    corr = X.corr()

    # --------------------------------------------------------
    # Publication-style formatting
    # --------------------------------------------------------
    sns.set_theme(
        style="white",
        font="serif",
        rc={
            "figure.dpi": dpi,
            "savefig.dpi": dpi,
            "axes.linewidth": 0.8,
            "font.size": 10,
            "axes.labelsize": 10,
            "xtick.labelsize": 6,
            "ytick.labelsize": 6,
        },
    )

    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        corr,
        ax=ax,
        cmap=cmap,
        vmin=-1,
        vmax=1,
        center=0,
        square=True,
        linewidths=0.0,
        cbar_kws={
            "shrink": 0.8,
            "label": "Correlation coefficient",
            "ticks": [-1, -0.5, 0, 0.5, 1],
        },
    )

    ax.set_title("Correlation Matrix of Topic Innovation Series", pad=12)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="both", which="both", length=0)

    plt.tight_layout()

    # save as high-res PNG
    plt.savefig(
        output_file,
        format="png",
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.02,
    )
    plt.close()

    print(f"Saved high-resolution PNG to: {output_file}")



In [ ]:
plot_corr_heatmap_png(X_innov, output_file="corr_topics_heatmap.png")


In [ ]:
import numpy as np
import pandas as pd
from itertools import product

def cross_lag_correlation_summary(df, lags=(1, 2, 3)):
    """
    Summarize correlations Corr(x_{i,t-k}, x_{j,t-l}) for i != j
    Vectorized implementation for speed.
    """
    X = df.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
    
    results = []
    
    for k, l in product(lags, lags):
        # Shift the data
        Xk = X.shift(k)
        Xl = X.shift(l)
        
        # Find rows where both are valid (no NaN in any column)
        max_lag = max(k, l)
        valid_mask = ~(Xk.isna().any(axis=1) | Xl.isna().any(axis=1))
        
        Xk_valid = Xk[valid_mask]
        Xl_valid = Xl[valid_mask]
        
        if len(Xk_valid) < 2:  # Need at least 2 observations
            results.append({
                "Lag k": k,
                "Lag l": l,
                "Mean corr": np.nan,
                "Median corr": np.nan,
                "Max |corr|": np.nan,
                "P95 |corr|": np.nan,
            })
            continue
        
        # Standardize the valid data
        Xk_std = (Xk_valid - Xk_valid.mean()) / Xk_valid.std()
        Xl_std = (Xl_valid - Xl_valid.mean()) / Xl_valid.std()
        
        # Compute correlation matrix
        n_valid = len(Xk_std)
        corr_matrix = (Xk_std.values.T @ Xl_std.values) / n_valid
        
        # Extract off-diagonal elements (i != j)
        n_cols = corr_matrix.shape[0]
        mask = ~np.eye(n_cols, dtype=bool)
        corrs = corr_matrix[mask]
        
        # Remove any remaining NaNs from the correlations
        corrs = corrs[~np.isnan(corrs)]
        
        if len(corrs) == 0:
            results.append({
                "Lag k": k,
                "Lag l": l,
                "Mean corr": np.nan,
                "Median corr": np.nan,
                "Max |corr|": np.nan,
                "P95 |corr|": np.nan,
            })
        else:
            results.append({
                "Lag k": k,
                "Lag l": l,
                "Mean corr": corrs.mean(),
                "Median corr": np.median(corrs),
                "Max |corr|": np.abs(corrs).max(),
                "P95 |corr|": np.percentile(np.abs(corrs), 95),
            })
    
    return pd.DataFrame(results)

In [ ]:
summary_df = cross_lag_correlation_summary(X_innov, lags=(1, 3, 5, 7, 9, 11))
print(summary_df.round(4))

In [7]:
# set seed
random.seed(42)

# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

# select a response variable (either marekt return or sp500 return)
y = response_variables['sprtrn']  # or 'sprtrn' for SP500 returns or vwretx

# transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# Final filtered dataframe
X = X[topic_cols]

def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

X = to_ar1_innovations(X)

# remove the first row wiht iloc
X = X.iloc[1:]

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [ ]:
# [I 2026-01-28 09:44:50,736] Trial 151 finished with value: 0.0039934730533537355 and parameters: {'window_size': 40, 'n_lags': 16, 'lambda': 0.00933750069285323}. Best is trial 151 with value: 0.0039934730533537355.


(1886,)

In [13]:
res  = estimate_single_config(
    X, y,
    window_size= 40,
    n_lags=16,
    lambda_val= 0.0093375,
    standardize=True,   
    verbose=True,
    return_details=True
)

In [14]:
res

{'summary': {'window_size': 40,
  'n_lags': 16,
  'lambda': 0.0093375,
  'r2_insample_stage1': np.float64(-0.007669100464297696),
  'r2_oos_stage1': np.float64(-0.008407567913079728),
  'r2_insample_stage2': np.float64(0.003498621825558379),
  'r2_oos_stage2': np.float64(-0.0034913158097733543),
  'kappa': np.float64(0.4183167212249348),
  'kappa_tstat': np.float64(4.348027182901111),
  'intercept': np.float64(0.00039622341550361005),
  'intercept_tstat': np.float64(1.7749777538873075),
  'n_observations': 1828,
  'n_windows': 1830,
  'n_oos_predictions_stage2': nan},
 'details':            date  window_size  n_lags    lambda  window_index  lasso_intercept  \
 0    2010-03-25           40      16  0.009338             0              0.0   
 1    2010-03-26           40      16  0.009338             1              0.0   
 2    2010-03-29           40      16  0.009338             2              0.0   
 3    2010-03-30           40      16  0.009338             3              0.0   
 4  

In [15]:
details = res["details"]

In [16]:
# count the rows with non-zero entries in details[num_nonzero_coefficients ]
num_nonzero = details["num_nonzero_coefficients"].value_counts()
num_nonzero

num_nonzero_coefficients
0.0     1731
1.0       34
2.0       22
8.0        9
6.0        8
3.0        5
7.0        5
5.0        4
4.0        4
9.0        4
10.0       3
12.0       1
11.0       1
Name: count, dtype: int64

In [ ]:
for i, val in res["details"]["num_nonzero_coefficients"].head(1500).items():
    print(f"Row {i}: num_nonzero_coefficients = {val}")


First Stage results

In [218]:
summaries

,window_size,n_lags,lambda,r2_insample_stage1,r2_oos_stage1,r2_insample_stage2,r2_oos_stage2,kappa,kappa_tstat,intercept,intercept_tstat,n_observations,n_windows,n_oos_predictions_stage2,target_column
0,350,8,0.002178,0.001883,-0.005641,1.628065e-03,0.000795,6.760195e-01,4.874177e+00,0.000748,2.803097,1528,1530,NaN,23393
1,350,8,0.002178,0.010245,-0.004135,6.467977e-05,-0.000039,1.369321e-01,3.640536e-01,0.000657,2.159537,1528,1530,NaN,67360
2,350,8,0.002178,0.220369,-0.053369,1.641755e-03,-0.003242,1.030654e-01,1.765618e+00,0.000389,0.674780,1528,1530,NaN,92635
3,350,8,0.002178,0.072856,-0.034114,3.856817e-04,0.000021,8.106094e-02,8.354103e-01,0.000716,1.828833,1528,1530,NaN,75341
4,350,8,0.002178,0.051321,-0.005407,1.473055e-03,-0.012999,2.164545e-01,1.914563e+00,0.000435,1.187827,1528,1530,NaN,92951
5,350,8,0.002178,0.072863,-0.017251,-1.025870e-08,-0.000590,5.997795e-06,3.694898e-05,0.000787,1.957350,1528,1530,NaN,89179
6,350,8,0.002178,0.048206,-0.015536,-7.269296e-12,-0.010950,1.378157e-09,7.000778e-09,0.000598,1.659994,1528,1530,NaN,92949
7,350,8,0.002178,0.021240,0.000064,5.022480e-03,-0.006023,4.720682e-01,5.261114e+00,0.000408,1.188642,1528,1530,NaN,80206
8,350,8,0.002178,0.420388,-0.092066,-7.805534e-12,-0.003459,1.211459e-09,2.113468e-08,-0.000517,-0.664798,1528,1530,NaN,83976
9,350,8,0.002178,0.400047,-0.144191,-4.799498e-09,-0.001112,2.111309e-07,3.951825e-06,-0.000202,-0.240045,1528,1530,NaN,43123


In [217]:
summaries = pd.read_parquet('stage1_stage2_summaries.parquet')
details = pd.read_parquet('stage1_stage2_details.parquet')

This code plots the average number of topics selected per window for each stock against the standard deviation of the stocks return in our sample. We should see a positive relationship.

In [ ]:
stock_cols = [col for col in stock_data.columns if str(col).isdigit()]
stock_data = stock_data[stock_cols]

# for column, calculate the standard deviation
std_devs = stock_data.std()

# calculate avg number of non-zero coefficients per stock
# identify LASSO coefficient columns (exclude intercept)
coef_cols = [c for c in details.columns if c.startswith("Lasso_")]

details["num_nonzero_recalc"] = (
    details[coef_cols]
    .notna()                # ignore NaNs
    .mul(details[coef_cols] != 0)
    .sum(axis=1)
)

# average per stock
avg_nonzero_df = (
    details
    .groupby("target_column")["num_nonzero_recalc"]
    .mean()
    .reset_index()
    .rename(columns={
        "target_column": "target_stock",
        "num_nonzero_recalc": "avg_nonzero_coefficients"
    })
)

plot_num_selected = avg_nonzero_df.merge(std_devs.rename("std_dev"), left_on="target_stock", right_index=True)

In [ ]:
import matplotlib.pyplot as plt
import os

# Output path
out_dir = r"C:\Users\jonat\Lasso_paper\Manuscripts\manuscript\figures"
os.makedirs(out_dir, exist_ok=True)

# Figure
plt.figure(figsize=(6, 4))

plt.scatter(
    plot_num_selected["std_dev"],
    plot_num_selected["avg_nonzero_coefficients"],
    s=18,
    alpha=0.6,
    color="black",
    edgecolor="none"
)

plt.xlabel("Stock Return Volatility", fontsize=11)
plt.ylabel("Average Number of Selected Predictors", fontsize=11)

plt.tick_params(axis="both", which="major", labelsize=10)
plt.grid(False)

plt.tight_layout()

# Save (journal-quality)
plt.savefig(os.path.join(out_dir, "avg_nonzero_coefficients_vs_volatility.pdf"))
plt.savefig(os.path.join(out_dir, "avg_nonzero_coefficients_vs_volatility.png"), dpi=300)


In [206]:
X.shape

(1886, 180)

First stage estimation results table

In [198]:
import pandas as pd
import numpy as np

# ---------- helper ----------
def summarize(s):
    return pd.Series({
        "mean":    s.mean(),
        "std-dev": s.std(),
        "median":  s.median(),
        "min":     s.min(),
        "max":     s.max(),
        "p05":     s.quantile(0.05),
        "p95":     s.quantile(0.95),
    })

# ---------- 1) R2s from summaries ----------
r2_in_stats  = summarize(summaries["r2_insample_stage1"])
r2_oos_stats = summarize(summaries["r2_oos_stage1"])

# ---------- 2) beta estimates (absolute non-zero LASSO coefficients) ----------
beta_cols = [c for c in details.columns if c.startswith("Lasso_")]
beta_vals = details[beta_cols].to_numpy().ravel()
beta_vals = beta_vals[beta_vals != 0]
beta_stats = summarize(pd.Series(np.abs(beta_vals)))

# ---------- 3) selection rate per stock (percent) ----------
sel_rate = (
    details.groupby("target_column")["num_nonzero_recalc"]
    .apply(lambda x: (x > 0).mean())
)
sel_rate_stats = summarize(sel_rate)

# ---------- final table ----------
out = pd.DataFrame(
    [r2_in_stats, r2_oos_stats, beta_stats, sel_rate_stats],
    index=[
        "r2_insample_stage1",
        "r2_oos_stage1",
        "beta_estimates",
        "selection_rate_percent",
    ],
)

print(out)



                            mean   std-dev    median           min       max  \
r2_insample_stage1      0.131942  0.158946  0.062089  1.882745e-03  0.420388   
r2_oos_stage1          -0.037165  0.047240 -0.016394 -1.441910e-01  0.000064   
beta_estimates          0.233294  0.287482  0.139433  8.446833e-07  3.936793   
selection_rate_percent  0.432681  0.420522  0.409654  0.000000e+00  0.998695   

                             p05       p95  
r2_insample_stage1      0.005646  0.411235  
r2_oos_stage1          -0.120735 -0.001825  
beta_estimates          0.009262  0.756294  
selection_rate_percent  0.000000  0.948500  


In [205]:
import pandas as pd
import numpy as np

# ---------- helper ----------
def summarize(s):
    return pd.Series({
        "mean":    s.mean(),
        "std-dev": s.std(),
        "median":  s.median(),
        "min":     s.min(),
        "max":     s.max(),
        "p05":     s.quantile(0.05),
        "p95":     s.quantile(0.95),
    })

# ---------- Stage 2 stats from summaries ----------
r2_in_2  = summarize(pd.to_numeric(summaries["r2_insample_stage2"], errors="coerce"))
r2_oos_2 = summarize(pd.to_numeric(summaries["r2_oos_stage2"], errors="coerce"))

kappa    = summarize(pd.to_numeric(summaries["kappa"], errors="coerce"))
kappa_t  = summarize(pd.to_numeric(summaries["kappa_tstat"], errors="coerce"))

# ---------- final table ----------
out_stage2 = pd.DataFrame(
    [r2_in_2, r2_oos_2, kappa, kappa_t],
    index=[
        "r2_insample_stage2",
        "r2_oos_stage2",
        "kappa",
        "kappa_tstat",
    ],
)

print(out_stage2)


                        mean   std-dev    median           min       max  \
r2_insample_stage2  0.001022  0.001580  0.000225 -1.025870e-08  0.005022   
r2_oos_stage2      -0.003760  0.004813 -0.002177 -1.299907e-02  0.000795   
kappa               0.168561  0.230584  0.092063  1.211459e-09  0.676020   
kappa_tstat         1.501498  2.013989  0.599732  7.000778e-09  5.261114   

                             p05       p95  
r2_insample_stage2 -7.802058e-09  0.003501  
r2_oos_stage2      -1.207703e-02  0.000447  
kappa               1.286473e-09  0.584241  
kappa_tstat         1.336103e-08  5.086993  
